# Build JSON alumpubs database from CSV file

In [1]:
import requests
import os
import json
from IPython.display import HTML
import pandas as pd
import bibtexparser
import re

In [4]:
CLIENT_ID = "APP-RL4PS7A03VARZ49I"
CLIENT_SECRET = "4266911b-37c7-4803-bf75-12afa11b6596"
REDIRECT_URL = "https://github.com/ASC-datascience/asc-alumpubs"


TOKEN_URL = "https://orcid.org/oauth/token"
API_BASE = "https://pub.orcid.org/v3.0"

## Functions

In [86]:

def get_public_token():
    r = requests.post(
        TOKEN_URL,
        headers={"Accept": "application/json"},
        data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "client_credentials",
            "scope": "/read-public",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["access_token"]


def get_json(url, token):
    r = requests.get(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

def get_work_summaries(orcid_id, token):
    return get_json(f"{API_BASE}/{orcid_id}/works", token)

def get_work_detail(orcid_id, put_code, token):
    return get_json(f"{API_BASE}/{orcid_id}/work/{put_code}", token)

def extract_put_codes(works_response):
    put_codes = []
    for group in works_response.get("group", []):
        for summary in group.get("work-summary", []):
            put_codes.append(summary["put-code"])
    return put_codes

def extract_citation(work):
    citation = work.get("citation")
    if not citation:
        return None

    return {
        "type": citation.get("citation-type"),
        "text": citation.get("citation-value"),
    }


def extract_doi(work):

    try:
        ids = work.get("external-ids", {}).get("external-id", [])
    except:
        ids = None 
        
    if not ids:
        return None
        
    for item in ids:
        if item.get("external-id-type") == "doi":
            return item.get("external-id-value")
    


def bibtex_from_crossref(doi):
    url = f"https://doi.org/{doi}"
    r = requests.get(
        url,
        headers={"Accept": "application/x-bibtex"},
        timeout=30,
    )
    if r.ok and r.text.strip():
        return r.text
    return None

In [35]:
def get_publications(ORCID, TOKEN):

    def pub_flatten(pdict):
        try:
            pubid = [*pdict][0]
            updated_dict = pdict[pubid]
            #updated_dict['pubid']=pubid
            return updated_dict
        except:
            return None

    
    print(f'Trying to get publications for {ORCID}')
    works = get_work_summaries(ORCID, TOKEN)
    pub_dict = []
    for put_code in extract_put_codes(works):
   
        work = get_work_detail(ORCID, put_code, TOKEN)
        doi = extract_doi(work)
        citation = bibtex_from_crossref(doi)
    
        if citation:
            citation = re.sub(r'(month=\w{3})\w','\\1', citation)

            bib_dict = bibtexparser.loads(citation).entries_dict

            
            
            item_dict = pub_flatten(bib_dict)
            
            pub_dict.append(item_dict)

    return pub_dict
    

In [6]:
# setup API token
TOKEN = get_public_token()

## 1. Load alumni data

In [46]:
alum_df = pd.read_csv('../data/alumni-scholar-database-v3.csv')

In [47]:
alum_df.columns

Index(['Name', 'AKA', 'Cohort', 'Graduation Year', 'Dissertation title',
       'Advisor', 'ORCHID ID', 'ORCHID Notes', 'Google Schoar ID',
       'Google Scholar Notes', 'Current institution', 'Website', 'Mentor?'],
      dtype='str')

In [48]:
colmap_dict = { cname: cname.replace('ORCHID','ORCID').lower().replace(' ','_').replace('?','')
                for cname in alum_df.columns  }

In [60]:
alum_df = alum_df.rename(columns=colmap_dict)
alum_df = alum_df.dropna(subset='name')
alum_df = alum_df.assign(cohort=alum_df['cohort'].astype(str), 
                         graduation_year=alum_df['graduation_year'].astype(str))

### Example structure for alumnus

```

{
      "id": "allyson-volinsky-2014",
      "full_name": "Allyson Volinsky",
      "aka": "Allyson Levin / Allyson Carol Volinsky",
      "cohort_year": 2014,
      "graduation_year": null,
      "website": null,
      "dissertation_title": "Variation on a theme: Comparing strategies for choosing health communication campaign message topics",
      "advisor": "Robert C. Hornik, Ph.D.",
      "publications": [
        {
          "title": "Affective Infrastructures and the Cultura Ipsum of Networked Care",
          "journal": "",
          "publication_year": 2017,
          "publication_type": "article",
          "doi": "",
          "url": "",
          "record_status": "synthetic_placeholder",
          "data_source": "temporary_fake_spreadsheet"
        },
```

In [27]:
alum_template = {
      "id": None,
      "full_name": None,
      "aka": None,
      "cohort_year": None,
      "graduation_year": None,
      "website": None,
      "dissertation_title": None,
      "advisor": None,
      "publications": []
}

In [75]:
def map_alumnus(item):
    alum_json = alum_template.copy()
    alum_json['full_name']=item['name']
    alum_json['cohort_year']=item['cohort']
    alum_json['id']=f"{alum_json['full_name'].lower().replace(' ','-')}-{alum_json['cohort_year']}"
    alum_json['aka']=item['aka']
    alum_json['graduation_year']=item['graduation_year']
    alum_json['website']=item['website']
    alum_json['dissertation_title']=item['dissertation_title']
    alum_json['advisor']=item['advisor']
    alum_json['orcid']=item['orcid_id']
    if type(item['orcid_id']) is str and item['orcid_id'].count('orcid.org'):
        orcid = item['orcid_id'].split('/')[-1]
        alum_json['publications']=get_publications(orcid, TOKEN)
    
    return alum_json

In [84]:
alum_df.loc[64]

name                                            Matt Lapierre
aka                                                       NaN
cohort                                                   2006
graduation_year                                           NaN
dissertation_title                                        NaN
advisor                                                   NaN
orcid_id                https://orcid.org/0000-0001-7098-0129
orcid_notes                                               NaN
google_schoar_id                                          NaN
google_scholar_notes                                      NaN
current_institution                                       NaN
website                                                   NaN
mentor                                                    NaN
Name: 64, dtype: object

In [88]:
json.dumps(map_alumnus(alum_df.loc[65]))
            

Trying to get publications for 0000-0002-2130-2441


'{"id": "nehama-lewis-persky-2006", "full_name": "Nehama Lewis-Persky", "aka": NaN, "cohort_year": "2006", "graduation_year": NaN, "website": NaN, "dissertation_title": NaN, "advisor": NaN, "publications": [{"pages": "1\\u201322", "month": "April", "year": "2026", "author": "Lewis, Nehama and Meir, Naama and Hetzroni, Orit E.", "publisher": "Informa UK Limited", "journal": "Journal of Applied Communication Research", "doi": "10.1080/00909882.2026.2648243", "url": "http://dx.doi.org/10.1080/00909882.2026.2648243", "issn": "1479-5752", "title": "How does neurodiversity impact information processing, communicative competence and task performance in video mediated interactions?", "ENTRYTYPE": "article", "ID": "Lewis_2026"}, {"pages": "1\\u201313", "month": "January", "year": "2026", "author": "Eliash-Fizik, Hadar and Lewis, Nehama and Sznitman, Sharon R.", "publisher": "Informa UK Limited", "journal": "Health Communication", "doi": "10.1080/10410236.2025.2610730", "url": "http://dx.doi.org

In [92]:
idx=list(alum_df.index)

In [91]:
alum_data = []

In [95]:
a=[1,2,32]
a.pop()

32

In [96]:
while i := idx.pop():
    try:
        print(f'Trying idx {i}')
        alum_data.append(map_alumnus(alum_df.loc[i]))
    except:
        print(f'ERROR idx {i}')

Trying idx 261
Trying to get publications for 0000-0002-6405-5137
Trying idx 260
Trying to get publications for 0000-0001-6581-3625
Trying idx 259
Trying idx 258
Trying idx 257
Trying idx 256
Trying to get publications for 0009-0002-9861-7932
Trying idx 255
Trying idx 254
Trying idx 253
Trying to get publications for 0000-0001-9685-2694
Trying idx 252
Trying idx 251
Trying to get publications for 0000-0002-1907-3344
Trying idx 250
Trying to get publications for 0000-0002-2190-8101
Trying idx 249
Trying idx 247
Trying to get publications for 0000-0002-1396-0778
Trying idx 246
Trying idx 245
Trying idx 244
Trying to get publications for 0000-0001-7597-0819
Trying idx 243
Trying to get publications for 0000-0003-3634-4449
Trying idx 242
Trying idx 241
Trying to get publications for 0000-0002-3465-0621
Trying idx 240
Trying to get publications for 0000-0002-9768-5733
Trying idx 239
Trying to get publications for 0000-0001-5208-3918
Trying idx 238
Trying to get publications for 0009-0002-47

IndexError: pop from empty list

In [97]:
len(alum_data)

237

In [79]:
alum_data = alum_df.apply(map_alumnus, axis=1)

Trying to get publications for 0000-0002-5005-0394
Trying to get publications for 0000-0003-3022-9706
Trying to get publications for 0009-0007-1848-1550
Trying to get publications for 0000-0002-2756-5197
Trying to get publications for 0000-0002-7608-1690
Trying to get publications for 0000-0001-5674-5595
Trying to get publications for 0000-0001-5526-1839
Trying to get publications for 0000-0003-4852-7373
Trying to get publications for 0000-0002-4510-7878
Trying to get publications for 0000-0001-9268-3969
Trying to get publications for 0000-0002-3613-7282
Trying to get publications for 0000-0002-3503-3289
Trying to get publications for 0000-0001-9584-7172
Trying to get publications for 0000-0001-6314-8027
Trying to get publications for 0000-0002-6514-801X
Trying to get publications for 0000-0001-7098-0129


AttributeError: 'NoneType' object has no attribute 'get'

In [80]:
alist = alum_data.to_list()

NameError: name 'alum_data' is not defined

In [81]:
len(alist)

NameError: name 'alist' is not defined

In [101]:
with open('../data/alumni-scholar-database-v3.json','w') as out:
    out.write(json.dumps(alum_data, indent=4))

------

## SCRATCHPAD

In [119]:
d={'a': {'x':1,'y':2}}

In [120]:
[v.update({'pubid': k}) for k, v in d.items()]

[None]

In [130]:
[*d][0]

'a'

In [142]:
pub_flatten(alist[0]['publications'][1])

{'pages': '98–103',
 'month': 'October',
 'year': '2020',
 'author': 'Levin, Allyson Volinsky',
 'publisher': 'Informa UK Limited',
 'journal': 'Communication Teacher',
 'number': '2',
 'doi': '10.1080/17404622.2020.1829666',
 'url': 'http://dx.doi.org/10.1080/17404622.2020.1829666',
 'issn': '1740-4630',
 'volume': '35',
 'title': 'What’s in a “welcome survey”? Designing a course welcome survey as introduction to research methods in Communication',
 'ENTRYTYPE': 'article',
 'ID': 'Levin_2020',
 'pubid': 'Levin_2020'}

In [147]:
for item in alist:
    pitems = []
    for pitem in item['publications']:
        pitems.append(pub_flatten(pitem))

    item['publications']=pitems

In [150]:
with open('alumni_pub_db.json','w') as out:
    out.write(json.dumps(alist, indent=4))

In [12]:
auth_url='https://sandbox.orcid.org/oauth/authorize'

In [14]:
MBOD_ORCID="0000-0001-5665-3930"

In [18]:
alum_df = pd.read_csv('../data/alumni-scholar-database-v3.csv')
alum_df.columns

Index(['Name', 'AKA', 'Cohort', 'Graduation Year', 'Dissertation title',
       'Advisor', 'ORCHID ID', 'ORCHID Notes', 'Google Schoar ID',
       'Google Scholar Notes', 'Current institution', 'Website'],
      dtype='str')

In [19]:
TOKEN = get_public_token()

In [20]:
oid=alum_df['ORCHID ID'][2].split('/')[-1]

In [86]:
oid = '0000-0002-6062-8487'
works = get_work_summaries(oid, TOKEN)
for put_code in extract_put_codes(works):
   
    work = get_work_detail(oid, put_code, TOKEN)
    doi = extract_doi(work)
    citation = bibtex_from_crossref(doi)

    print(citation)

    citation = re.sub(r'(month=\w{3})\w','\\1', citation)
    
    print(bibtexparser.loads(citation).entries_dict)
    print('---')

 @article{Siegel_2020, title={Do Longitudinal Trends in Tobacco 21-Related Media Coverage Correlate with Policy Support? an Exploratory Analysis Using Supervised and Unsupervised Machine Learning Methods}, volume={37}, ISSN={1532-7027}, url={http://dx.doi.org/10.1080/10410236.2020.1816282}, DOI={10.1080/10410236.2020.1816282}, number={1}, journal={Health Communication}, publisher={Informa UK Limited}, author={Siegel, Leeann N. and Levin, Allyson Volinsky and Kranzler, Elissa C. and Gibson, Laura A.}, year={2020}, month=Sept, pages={29–38} }

{'Siegel_2020': {'pages': '29–38', 'month': 'September', 'year': '2020', 'author': 'Siegel, Leeann N. and Levin, Allyson Volinsky and Kranzler, Elissa C. and Gibson, Laura A.', 'publisher': 'Informa UK Limited', 'journal': 'Health Communication', 'number': '1', 'doi': '10.1080/10410236.2020.1816282', 'url': 'http://dx.doi.org/10.1080/10410236.2020.1816282', 'issn': '1532-7027', 'volume': '37', 'title': 'Do Longitudinal Trends in Tobacco 21-Related 

In [23]:
t='''
 @article{Fernandes_2026, title={The Influence of Emotion Dynamics on Interpersonal Liking}, url={http://dx.doi.org/10.31234/osf.io/yb3hx_v1}, DOI={10.31234/osf.io/yb3hx_v1}, publisher={Center for Open Science}, author={Fernandes, Laura Furtado and Ford, Ezra and Baek, Elisa C and Burns, Shannon M.}, year={2026}, month=Apr }

 @article{Lu_2025, title={Idiosyncratic Event Segmentation as a Neural Marker of Loneliness}, url={http://dx.doi.org/10.31234/osf.io/7rbhy_v1}, DOI={10.31234/osf.io/7rbhy_v1}, publisher={Center for Open Science}, author={Lu, Chang and Sava-Segal, Clara and Baek, Elisa C}, year={2025}, month=Nov }

 @article{Ma_de_Sousa_2025, title={Loneliness is associated with unstable and distorted emotion transition predictions}, volume={3}, ISSN={2731-9121}, url={http://dx.doi.org/10.1038/s44271-025-00310-w}, DOI={10.1038/s44271-025-00310-w}, number={1}, journal={Communications Psychology}, publisher={Springer Science and Business Media LLC}, author={Ma de Sousa, Ava Q. and Schwyck, Miriam E. and Furtado Fernandes, Laura and Ford, Ezra and Babür, Begüm G. and Lu, Chang and Zimmerman, Jacob C. and Yu, Hongbo and Burns, Shannon M. and Baek, Elisa C.}, year={2025}, month=Aug }

 @article{Baek_2025, title={Perceived community alignment increases information sharing}, volume={16}, ISSN={2041-1723}, url={http://dx.doi.org/10.1038/s41467-025-59915-8}, DOI={10.1038/s41467-025-59915-8}, number={1}, journal={Nature Communications}, publisher={Springer Science and Business Media LLC}, author={Baek, Elisa C. and Hyon, Ryan and López, Karina and Porter, Mason A. and Parkinson, Carolyn}, year={2025}, month=July }

 @article{Baek_2025, title={The four conceptualizations of social connection}, volume={4}, ISSN={2731-0574}, url={http://dx.doi.org/10.1038/s44159-025-00455-9}, DOI={10.1038/s44159-025-00455-9}, number={8}, journal={Nature Reviews Psychology}, publisher={Springer Science and Business Media LLC}, author={Baek, Elisa C. and Pourafshari, Razieh and Bayer, Joseph B.}, year={2025}, month=June, pages={506–517} }

 @article{Ma_de_Sousa_2025, title={Loneliness is associated with unstable and distorted emotion transition predictions}, url={http://dx.doi.org/10.31234/osf.io/6g4sc_v2}, DOI={10.31234/osf.io/6g4sc_v2}, publisher={Center for Open Science}, author={Ma de Sousa, Ava Q. and Schwyck, Miriam E. and Fernandes, Laura Furtado and Ford, Ezra and Babur, Begum and Lu, Chang and Zimmerman, Jacob and Yu, Hongbo and Burns, Shannon M. and Baek, Elisa C}, year={2025}, month=Mar }

 @article{Ma_de_Sousa_2025, title={Loneliness is associated with unstable and distorted emotion transition predictions}, url={http://dx.doi.org/10.31234/osf.io/6g4sc_v1}, DOI={10.31234/osf.io/6g4sc_v1}, publisher={Center for Open Science}, author={Ma de Sousa, Ava Q. and Schwyck, Miriam E. and Fernandes, Laura Furtado and Ford, Ezra and Babur, Begum and Lu, Chang and Zimmerman, Jacob and Yu, Hongbo and Burns, Shannon M. and Baek, Elisa C}, year={2025}, month=Mar }

 @article{Baek_2025, title={Having more friends is associated with greater sensitization to social exclusion: neural and behavioral evidence}, volume={20}, ISSN={1749-5024}, url={http://dx.doi.org/10.1093/scan/nsaf067}, DOI={10.1093/scan/nsaf067}, number={1}, journal={Social Cognitive And Affective Neuroscience}, publisher={Oxford University Press (OUP)}, author={Baek, Elisa C and Shen, Yixuan Lisa and Kim, Hairin and Baldina, Ekaterina and Chey, Jeanyung and Youm, Yoosik and Parkinson, Carolyn}, year={2025} }

 @article{Baek_2023, title={Lonely Individuals Process the World in Idiosyncratic Ways}, volume={34}, ISSN={1467-9280}, url={http://dx.doi.org/10.1177/09567976221145316}, DOI={10.1177/09567976221145316}, number={6}, journal={Psychological Science}, publisher={SAGE Publications}, author={Baek, Elisa C. and Hyon, Ryan and López, Karina and Du, Meng and Porter, Mason A. and Parkinson, Carolyn}, year={2023}, month=Apr, pages={683–695} }

 @article{Abdurahman_2023, title={Targeting audiences’ moral values shapes misinformation sharing}, url={http://dx.doi.org/10.31234/osf.io/ztq2k}, DOI={10.31234/osf.io/ztq2k}, publisher={Center for Open Science}, author={Abdurahman, Suhaib and Reimer, Nils Karl and Golazizian, Preni and Baek, Elisa C and Shen, Yixuan and Trager, Jackson and Lulla, Roshni and Kaplan, Jonas and Parkinson, Carolyn and Dehghani, Morteza}, year={2023}, month=May }

 @article{Baek_2023, title={Perceived community alignment increases information sharing}, url={http://dx.doi.org/10.31234/osf.io/vea3k}, DOI={10.31234/osf.io/vea3k}, publisher={Center for Open Science}, author={Baek, Elisa C and Hyon, Ryan and López, Karina and Porter, Mason A. and Parkinson, Carolyn}, year={2023}, month=Apr }

 @article{Baek_2022, title={Shared understanding and social connection: Integrating approaches from social psychology, social network analysis, and neuroscience}, volume={16}, ISSN={1751-9004}, url={http://dx.doi.org/10.1111/spc3.12710}, DOI={10.1111/spc3.12710}, number={11}, journal={Social and Personality Psychology Compass}, publisher={Wiley}, author={Baek, Elisa C. and Parkinson, Carolyn}, year={2022}, month=Oct }

 @article{Chan_2022, title={The gap between sharing and reading news on social media: A multi-method investigation}, url={http://dx.doi.org/10.31234/osf.io/65qwb}, DOI={10.31234/osf.io/65qwb}, publisher={Center for Open Science}, author={Chan, Hang Yee and Scholz, Christin and Baek, Elisa C and Falk, Emily B.}, year={2022}, month=June }

 @article{Baek_2022, title={In-degree centrality in a social network is linked to coordinated neural activity}, volume={13}, ISSN={2041-1723}, url={http://dx.doi.org/10.1038/s41467-022-28432-3}, DOI={10.1038/s41467-022-28432-3}, number={1}, journal={Nature Communications}, publisher={Springer Science and Business Media LLC}, author={Baek, Elisa C. and Hyon, Ryan and López, Karina and Finn, Emily S. and Porter, Mason A. and Parkinson, Carolyn}, year={2022}, month=Mar }

 @article{Chan_2021, title={Being the Gatekeeper: How Thinking about Sharing Affects Neural Encoding of Information}, volume={31}, ISSN={1460-2199}, url={http://dx.doi.org/10.1093/cercor/bhab060}, DOI={10.1093/cercor/bhab060}, number={8}, journal={Cerebral Cortex}, publisher={Oxford University Press (OUP)}, author={Chan, Hang-Yee and Scholz, Christin and Baek, Elisa C and O’Donnell, Matthew B and Falk, Emily B}, year={2021}, month=Apr, pages={3939–3949} }

 @article{Baek_2021, title={Lonely individuals process the world in idiosyncratic ways}, url={http://dx.doi.org/10.31234/osf.io/yt872}, DOI={10.31234/osf.io/yt872}, publisher={Center for Open Science}, author={Baek, Elisa C and Hyon, Ryan and López, Karina and Du, Meng and Porter, Mason A. and Parkinson, Carolyn}, year={2021}, month=July }

 @article{Baek_2021, title={Popular individuals process the world in particularly normative ways}, url={http://dx.doi.org/10.31234/osf.io/6fj2p}, DOI={10.31234/osf.io/6fj2p}, publisher={Center for Open Science}, author={Baek, Elisa C and Hyon, Ryan and López, Karina and Finn, Emily S. and Porter, Mason A. and Parkinson, Carolyn}, year={2021}, month=June }

 @article{Baek_2020, title={How do our brains support our real-life friendships?}, url={http://dx.doi.org/10.31235/osf.io/vy2jr}, DOI={10.31235/osf.io/vy2jr}, publisher={Center for Open Science}, author={Baek, Elisa C and Hyon, Ryan and Porter, Mason A. and Parkinson, Carolyn}, year={2020}, month=Dec }

 @article{Baek_2019, title={Social Network Analysis for Social Neuroscientists}, url={http://dx.doi.org/10.31234/osf.io/kgc2h}, DOI={10.31234/osf.io/kgc2h}, publisher={Center for Open Science}, author={Baek, Elisa C and Porter, Mason A. and Parkinson, Carolyn}, year={2019}, month=Sept }

 @article{Baek_2019, title={Considering Others’ Mental States Causally Increases Feelings of Social Bonding and Information Sharing}, url={http://dx.doi.org/10.31234/osf.io/nw43x}, DOI={10.31234/osf.io/nw43x}, publisher={Center for Open Science}, author={Baek, Elisa C and Tamir, Diana and Falk, Emily B.}, year={2019}, month=Sept }
 '''

In [39]:
t2='''@article{Baek_2019, title={Considering Others’ Mental States Causally Increases Feelings of Social Bonding and Information Sharing}, url={http://dx.doi.org/10.31234/osf.io/nw43x}, DOI={10.31234/osf.io/nw43x}, publisher={Center for Open Science}, author={Baek, Elisa C and Tamir, Diana and Falk, Emily B.}, year={2019}, month=Sep }'''

In [40]:
db=bibtexparser.loads(t2)

In [41]:
db.entries_dict

{'Baek_2019': {'month': 'September',
  'year': '2019',
  'author': 'Baek, Elisa C and Tamir, Diana and Falk, Emily B.',
  'publisher': 'Center for Open Science',
  'doi': '10.31234/osf.io/nw43x',
  'url': 'http://dx.doi.org/10.31234/osf.io/nw43x',
  'title': 'Considering Others’ Mental States Causally Increases Feelings of Social Bonding and Information Sharing',
  'ENTRYTYPE': 'article',
  'ID': 'Baek_2019'}}

In [29]:
import bibtexparser
library = bibtexparser.parse_string(t)

AttributeError: module 'bibtexparser' has no attribute 'parse_string'

In [31]:
bibtexparser.bparser.parse(t)

UndefinedString: 'july'